In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json
import matplotlib.colors as colors
# import holoviews as hv
from matplotlib.patches import Rectangle

from cycler import cycler
import matplotlib as mpl

In [ ]:
from scipy.signal import savgol_filter
import gzip
import matplotlib as mpl
from cycler import cycler
# hv.extension('bokeh')
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from scipy.signal import peak_widths
from numpy.fft import rfft, rfftfreq

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.gridspec import GridSpec

from scipy.signal import find_peaks
from scipy.signal import peak_widths
from scipy.optimize import curve_fit

import matplotlib.lines as mlines

import skunk

plt.rcParams.update({'font.size': 14})

In [ ]:
def load1dFig3(i):
    # return np.loadtxt(f'G:\\Shared drives\\GGG GDrive\\Pink Fridge Data\\20231026_HMIA13_6 QD\\data\\{i}\\data.tsv')
    return np.loadtxt(f'../20231026_HMIA13_6 QD/data/{i}/data.tsv')

def load2dFig3(i, num):
    tmp = np.loadtxt(f'../20231026_HMIA13_6 QD/data/{i}/data.tsv')
    numrows = np.floor(tmp.shape[0] / num)
    data_dict = {}
    for j in range(tmp.shape[1]):
        data_dict[str(j)] = tmp[:num*int(numrows), j].reshape((int(numrows), num)).transpose()
    return data_dict
# fast axis is column-wise

In [ ]:
def cbsinh(Vg, center, Vt0, x):
    deltaVg = Vg-center
    return Vt0*((1e-6+deltaVg*x)/(1e-6+np.sinh(deltaVg*x)))

import lmfit
from lmfit.models import Model

cbmodel = Model(cbsinh)
params = cbmodel.make_params()

In [ ]:
import glob
import os

# search_dir = 'G:\\Shared drives\\GGG GDrive\\Pink Fridge Data\\20231026_HMIA13_6 QD\\data\\'
# lia_dir = '/Users/praveen/Library/CloudStorage/GoogleDrive-prvn@stanford.edu/Shared drives/GGG GDrive/Pink Fridge Data/20231026_HMIA13_6 QD/data/lia/'
lia_dir = '../20231026_HMIA13_6 QD/data/lia/'
# remove anything from the list that is not a file (directories, symlinks)
# thanks to J.F. Sebastion for pointing out that the requirement was a list
# of files (presumably not including directories)
#files = list(filter(os.path.isfile, glob.glob(search_dir + "*")))
#files.sort(key=lambda x: os.path.getmtime(x))

def sortedfilelist(lia_dir):
    os.chdir(lia_dir)
    lia_dir_abs = os.getcwd()
    files =os.listdir(lia_dir_abs)
    files.sort(key=lambda x: os.path.getmtime(x))
    os.chdir('../../../figure4/')
    return files

In [ ]:
def noisepsd(dat, fs, rms=True):
    dat = dat - np.mean(dat)
    datfft = rfft(dat)
    fftfreq = rfftfreq(len(dat), 1/fs)
    if rms:
        scale = 2
    else:
        scale = np.sqrt(2)

    inoise_psd = 1*((len(dat)/fs)*(1*(1/(25813/3))*((scale/len(dat)*np.abs(datfft)))/1)**2)# A^2/Hz

    return fftfreq, inoise_psd

In [ ]:
fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()
# ax3 = ax2.twiny()
dat10 =  np.loadtxt(f'../20231026_HMIA13_6 QD/data//lia/' + '231102_SR860noisefloor_fs300_tc3ms_FIRfilter.txt.gz')

firstfile = '231107_1000s_syncOFF_liaf88Hz_noiseonleftCBF_rplunger_fs300_tc3ms_FIRfilter_5nAlia860only_Vgm502.8mVR_iter0.txt.gz'
numfiles = 20
filelist = sortedfilelist(lia_dir)
startindex = filelist.index(firstfile)
fs = 305.18
alpha = 0.18/6
slope = 0.0017
time = np.arange(0, 1000, 1/fs)
chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
counter = 0
chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
for peaknum in range(numfiles):


    if peaknum%1==0:
        dat = np.loadtxt('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum])
        sigma = np.std(dat)

        with gzip.open('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum],'r') as f:
            firstline = f.readline()
            header = str(firstline).split(',')
            slope = float(header[3])
            print(slope)
        f.close()

        if 1e9*sigma < 300:
            flag=1
            counter+=1
        else:
            flag=0

        ax.plot(time[:len(dat)], dat+peaknum*1*5e-6, label='peaknum = %1.f, $\sigma = $ %2.f nV' %(peaknum, 1e9*sigma))
        print(1e9*sigma)
        print(flag)
        print('--')
        f, noise = noisepsd(dat, fs, rms=True)

        chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
        chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])

print(f'Counter = {counter}')
avgchargenoise_left = chargenoisesum/counter
# ax2.semilogy(f, chargenoisesum/counter, ls='--', color='b')



firstfile = '231107_1000s_syncOFF_liaf88Hz_noiseonrightCBF_rplunger_fs300_tc3ms_FIRfilter_5nAlia860only_Vgm499.6mVR_iter0.txt.gz'
numfiles = 20
filelist = sortedfilelist(lia_dir)
startindex = filelist.index(firstfile)
fs = 305.18
alpha = 0.18/6
slope = 0.0017
time = np.arange(0, 1000, 1/fs)
chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
counter = 0
chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
for peaknum in range(numfiles):


    if peaknum%1==0:
        dat = np.loadtxt('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum])
        sigma = np.std(dat)

        with gzip.open('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum],'r') as f:
            firstline = f.readline()
            header = str(firstline).split(',')
            slope = float(header[3])
            print(slope)
        f.close()

        if 1e9*sigma < 300:
            flag=1
            counter+=1
        else:
            flag=0

        ax.plot(time[:len(dat)], dat+peaknum*1*5e-6, label='peaknum = %1.f, $\sigma = $ %2.f nV' %(peaknum, 1e9*sigma))
        print(1e9*sigma)
        print(flag)
        print('--')
        f, noise = noisepsd(dat, fs, rms=True)

        chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
        chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])


avgchargenoise_right = chargenoisesum/counter
# ax2.semilogy(f, chargenoisesum/counter, ls='--', color='r')


# firstfile = '231107_1000s_syncOFF_liaf88Hz_noiseonleftCBF_rplunger_fs300_tc3ms_FIRfilter_5nAlia860_Vgm500.6mVR_iter0.txt.gz'
# numfiles = 10
# filelist = sortedfilelist(lia_dir)
# startindex = filelist.index(firstfile)
# fs = 305.18
# alpha = 0.18/6
# slope = 1e-3
# time = np.arange(0, 1000, 1/fs)
# chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
# counter = 0
# chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
# for peaknum in range(numfiles):


#     if peaknum%1==0:
#         dat = np.loadtxt(filelist[startindex+peaknum])
#         sigma = np.std(dat)

#         if 1e9*sigma < 1000:
#             flag=1
#             counter+=1
#         else:
#             flag=0

#         ax.plot(time[:len(dat)], dat+peaknum*1*5e-6, label='peaknum = %1.f, $\sigma = $ %2.f nV' %(peaknum, 1e9*sigma))
#         print(1e9*sigma)
#         print(flag)
#         print('--')
#         f, noise = noisepsd(dat, fs, rms=True)

#         chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
#         chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])



# ax2.semilogy(f, chargenoisesum/counter, ls='--', color='k')
firstfile = '231108_1000s_syncOFF_liaf88Hz_noiseonCBV_rplunger_fs300_tc3ms_FIRfilter_5nAlia860only_Vgm500.0mVR_iter0.txt.gz'
numfiles = 1
filelist = sortedfilelist(lia_dir)
startindex = filelist.index(firstfile)
fs = 305.18
alpha = 0.18/6
slope = 0.0017
time = np.arange(0, 1000, 1/fs)
chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
counter = 0
chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
for peaknum in range(numfiles):


    if peaknum%1==0:
        dat = np.loadtxt('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum])
        sigma = np.std(dat)

        if 1e9*sigma < 300:
            flag=1
            counter+=1
        else:
            flag=0

        ax.plot(time[:len(dat)], dat+peaknum*1*5e-6, label='peaknum = %1.f, $\sigma = $ %2.f nV' %(peaknum, 1e9*sigma))
        print(1e9*sigma)
        print(flag)
        print('--')
        f, noise = noisepsd(dat, fs, rms=True)

        chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
        chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])


cbv = chargenoisesum/counter


firstfile = '231108_1000s_syncOFF_liaf88Hz_noiseonCBP_rplunger_fs300_tc3ms_FIRfilter_5nAlia860only_Vgm504.3mVR_iter0.txt.gz'
numfiles = 1
filelist = sortedfilelist(lia_dir)
startindex = filelist.index(firstfile)
fs = 305.18
alpha = 0.18/6
slope = 0.0017
time = np.arange(0, 1000, 1/fs)
chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
counter = 0
chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
for peaknum in range(numfiles):


    if peaknum%1==0:
        dat = np.loadtxt('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum])
        sigma = np.std(dat)

        if 1e9*sigma < 300:
            flag=1
            counter+=1
        else:
            flag=0

        ax.plot(time[:len(dat)], dat+peaknum*1*5e-6, label='peaknum = %1.f, $\sigma = $ %2.f nV' %(peaknum, 1e9*sigma))
        print(1e9*sigma)
        print(flag)
        print('--')
        f, noise = noisepsd(dat, fs, rms=True)

        chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
        chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])


cbp = chargenoisesum/counter

ax2.loglog(f[1:], 2e-13/f[1:]**1, ls='--', label='$1/f$', color='b')
ax2.loglog(f[1:200], 2e-15/f[1:200]**2, ls='--', label='$1/f^2$', color='r')
fnoisefloor300, noisefloor300 =  noisepsd(dat10, fs, rms=True)
chargenoisefloor300 = noisefloor300*((25813/3)**2)*(alpha**2)/(slope**2)

ax2.set_ylim(1e-16, 1e-6)
ax2.set_xscale('linear')
# ax2.set_xlim(2, 5)
ax2.set_xlabel('frequency (Hz)')
ax2.set_ylabel('S$_{\mu}$ (eV$^2$/Hz)')
ax2.grid(ls='--', lw=0.4, which='both')
ax2.legend()

In [ ]:
fig, ax = plt.subplots()
vgpeak = np.zeros((500,))
time = np.linspace(28, 28*500, 500)
for i,d in enumerate(np.arange(988, 1488,1)):
    dat  = load1dFig3(d)
    vt = dat[:,2]
    vg = dat[:,1]
    vtfilter = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg)[0])
    maxidx = np.argmax(vtfilter)
    vgpeak[i] = vg[maxidx]
ax.plot(time, vgpeak)
    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)

In [ ]:
fs = 1/28
flow, noiseCPT = noisepsd(vgpeak, fs, rms=False)

In [ ]:
fig, ax = plt.subplots()
vgpeak = np.zeros((1000,))
time = np.linspace(28, 28*1000, 1000)
for i,d in enumerate(np.arange(1488, 2488,1)):
    dat  = load1dFig3(d)
    vt = dat[:,2]
    vg = dat[:,1]
    vtfilter = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg)[0])
    maxidx = np.argmax(vtfilter)
    vgpeak[i] = vg[maxidx]
ax.plot(time, vgpeak)
    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)

fs = 1/28
flow2, noiseCPT2 = noisepsd(vgpeak, fs, rms=False)

In [ ]:
fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()
b = np.linspace(0,1,1001)
# ax2 = ax.twinx()
vgpeak = np.zeros((500,))
peak_temps = np.zeros((500,))
peak_temps_err = np.zeros((500,))
time = np.linspace(41, 41*500, 500)

alpha = 0.18/6 ## lever arm
kT = 4.05e-6
k = 8.6e-5
for i,d in enumerate(np.arange(2506, 3506,2)):
    dat  = load1dFig3(d)
    vt = dat[:,2]
    vg = dat[:,1]


    # ax.plot(vg, vt)
    vtfilter = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg)[0])
    maxidx = np.argmax(vtfilter)

    if i%10==0:
        ax2.plot(vg, vtfilter, lw=0.5, color='k')
        ax2.plot(vg[maxidx], vtfilter[maxidx], 'rx')

    params['center'].set(value = vg[maxidx], min = vg[0], max = vg[-1])
    params['Vt0'].set(value = vt[maxidx], min = 0, max = 10*vt[maxidx])
    params['x'].set(value=alpha/kT, min = 0, max = 10*alpha/kT)

    result = cbmodel.fit(vt, params, Vg = vg)
    peak_temps[i] = alpha/(k*result.params['x'].value)
    peak_temps_err[i] = peak_temps[i]*result.params['x'].stderr/result.params['x'].value
    vgpeak[i] = result.params['center'].value

ax.plot(time[:i]/3600, vgpeak[:i])
ax.set_ylabel('$V_{peak}$ (V)')
ax.set_xlabel('time (hours)')
# ax2.errorbar(time[:i], 1e3*peak_temps[:i], yerr = 1e3*peak_temps_err[:i], fmt = 'r.')

    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)

fs = 1/41
flow3, noiseCPT3 = noisepsd(vgpeak[:i], fs, rms=False)

In [ ]:
fig, ax = plt.subplots()
ax2 = ax.twinx()
vgpeak = np.zeros((500,))
peak_temps = np.zeros((500,))
peak_temps_err = np.zeros((500,))
time = np.linspace(41, 41*500, 500)

alpha = 0.18/6 ## lever arm
kT = 4.05e-6
k = 8.6e-5
for i,d in enumerate(np.arange(3508, 3808,2)):
    dat  = load1dFig3(d)
    vt = dat[:,2]
    vg = dat[:,1]


    # ax.plot(vg, vt)
    vtfilter = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg)[0])
    maxidx = np.argmax(vtfilter)


    params['center'].set(value = vg[maxidx], min = vg[0], max = vg[-1])
    params['Vt0'].set(value = vt[maxidx], min = 0, max = 10*vt[maxidx])
    params['x'].set(value=alpha/kT, min = 0, max = 10*alpha/kT)

    result = cbmodel.fit(vt, params, Vg = vg)
    peak_temps[i] = alpha/(k*result.params['x'].value)
    peak_temps_err[i] = peak_temps[i]*result.params['x'].stderr/result.params['x'].value
    vgpeak[i] = result.params['center'].value

ax.plot(time[:i], vgpeak[:i])
ax2.errorbar(time[:i], 1e3*peak_temps[:i], yerr = 1e3*peak_temps_err[:i], fmt = 'r.')

    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)
#NOTE: flow4 corresponds to guess_plunger = -0.518 V
fs = 1/41
flow4, noiseCPT4 = noisepsd(vgpeak[:i], fs, rms=False)

In [ ]:
fig, ax = plt.subplots()
fig2, ax2 = plt.subplots()
# ax2 = ax.twinx()
numtraces = 100
peaksetrange = int(np.floor((10261 - 3812)/(2*numtraces)))
vgpeak = np.zeros((numtraces,))
peak_temps = np.zeros((numtraces,))
peak_temps_err = np.zeros((numtraces,))
time = np.linspace(41, 41*numtraces, numtraces)
fs = 1/41
alpha = 0.18/6 ## lever arm
kT = 4.05e-6
k = 8.6e-5
noiseCPT5 = np.zeros((int(fs/2*41*numtraces), peaksetrange))
for peakset in range(peaksetrange):
    print(peakset)
    for i,d in enumerate(np.arange(3812 + 2*numtraces*peakset, 3812 + 2*numtraces*(peakset+1),2)):
        dat  = load1dFig3(d)
        vt = dat[:,2]
        vg = dat[:,1]


        # ax.plot(vg, vt)
        vtfilter = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg)[0])
        maxidx = np.argmax(vtfilter)


        params['center'].set(value = vg[maxidx], min = vg[0], max = vg[-1])
        params['Vt0'].set(value = vt[maxidx], min = 0, max = 10*vt[maxidx])
        params['x'].set(value=alpha/kT, min = 0, max = 10*alpha/kT)

        result = cbmodel.fit(vt, params, Vg = vg)
        peak_temps[i] = alpha/(k*result.params['x'].value)
        # peak_temps_err[i] = peak_temps[i]*result.params['x'].stderr/result.params['x'].value
        vgpeak[i] = result.params['center'].value

    flow5, noiseCPT5[:,peakset] = noisepsd(vgpeak[:i], fs, rms=False)

    ax.plot(time[:i], vgpeak[:i])

chargenoiseCPT = np.mean(noiseCPT5*((25813/3)**2)*(alpha**2), axis=1)

ax2.loglog(flow5, chargenoiseCPT)
# ax2.plot(time[:i], 1e3*peak_temps[:i])

    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)

In [ ]:
import os
import io


def safe_svg2pdf(svg: str, output_path: str = None) -> bytes:
    """
    Converts SVG string to PDF and writes to output_path if provided.
    Returns PDF as bytes either way.

    Args:
        svg (str): SVG content.
        output_path (str, optional): Path to write PDF file. Can include ~.

    Returns:
        bytes: The resulting PDF file content.
    """
    buffer = io.BytesIO()

    try:
        # Write to in-memory buffer first
        cairosvg.svg2pdf(bytestring=svg, write_to=buffer)
        pdf_data = buffer.getvalue()

        if output_path:
            # Expand ~ and write manually to file
            expanded_path = os.path.expanduser(output_path)
            with open(expanded_path, 'wb') as f:
                f.write(pdf_data)
            print(f"✅ PDF written to: {expanded_path}")
        else:
            print("✅ PDF generated in memory.")

        return pdf_data

    except OSError as e:
        print(f"❌ CairoSVG failed: {e}")
        return b''  # Or raise if you'd prefer

import os
import io


def safe_svg2pdf(svg: str, output_path: str = None) -> bytes:
    """
    Converts SVG string to PDF and writes to output_path if provided.
    Returns PDF as bytes either way.

    Args:
        svg (str): SVG content.
        output_path (str, optional): Path to write PDF file. Can include ~.

    Returns:
        bytes: The resulting PDF file content.
    """
    buffer = io.BytesIO()

    try:
        # Write to in-memory buffer first
        cairosvg.svg2pdf(bytestring=svg, write_to=buffer)
        pdf_data = buffer.getvalue()

        if output_path:
            # Expand ~ and write manually to file
            expanded_path = os.path.expanduser(output_path)
            with open(expanded_path, 'wb') as f:
                f.write(pdf_data)
            print(f"✅ PDF written to: {expanded_path}")
        else:
            print("✅ PDF generated in memory.")

        return pdf_data

    except OSError as e:
        print(f"❌ CairoSVG failed: {e}")
        return b''  # Or raise if you'd prefer



In [ ]:
from scipy.signal import firwin, lfilter, freqz

In [ ]:
plt.rcParams.update({'font.size': 14})
fig3 = plt.figure(figsize=(16, 8),constrained_layout=True)
gs = fig3.add_gridspec(8, 4, width_ratios=(1,1, 1, 1))
# gs = GridSpec(3, 3, figure=fig1)
f3_ax1 = fig3.add_subplot(gs[:4, :2])
# f3_ax1.set_title('gs[0, :3]')
f3_ax2 = fig3.add_subplot(gs[:4, 2:3])
f3_ax5 = fig3.add_subplot(gs[4:8, :2], projection='3d')
f3_ax4 = fig3.add_subplot(gs[:4, 3])
# inset_ax = fig3.add_subplot(gs[:1, 2])
# f1_ax2cbar = fig1.add_subplot(gs[0, 3])
# plt.setp(f4_ax2.get_yticklabels(), visible=False)
# f3_ax2.set_title('gs[0, 3:]')
f3_ax3 = fig3.add_subplot(gs[4:8,2:4])


# Define style dictionary
style_dict = {
    'color': 'blue',
    'linewidth': 2,
    'fontsize': 14
}

# Apply style
spine = f3_ax2.spines['left']
spine.set_color(style_dict['color'])
spine.set_linewidth(style_dict['linewidth'])

f3_ax2.tick_params(axis='y', colors=style_dict['color'])
# f3_ax2.set_xlabel('X Axis', color=style_dict['color'], fontsize=style_dict['fontsize'])
f3_ax2.set_ylabel(r'$G$($e^2/h$)', color=style_dict['color'], fontsize=style_dict['fontsize'])


ax2 = f3_ax2.twinx()
ax2.annotate('', xytext=(1, 25), xy=(2, 25),
            arrowprops=dict(color='r', arrowstyle="->", ls='--'))
# Define style dictionary
style_dict = {
    'color': 'red',
    'linewidth': 2,
    'fontsize': 14
}

# Apply style
spine = ax2.spines['right']
spine.set_color(style_dict['color'])
spine.set_linewidth(style_dict['linewidth'])

ax2.tick_params(axis='y', colors=style_dict['color'])
# f3_ax2.set_xlabel('X Axis', color=style_dict['color'], fontsize=style_dict['fontsize'])
ax2.set_ylabel('Y Axis', color=style_dict['color'], fontsize=style_dict['fontsize'])



f3_ax1.text(-0.1, 1, "(a)", fontsize=14, va="bottom", ha="right", transform=f3_ax1.transAxes)
f3_ax1.set_axis_off()
f3_ax2.text(-0.1, 1, "(b)", fontsize=14, va="bottom", ha="right", transform=f3_ax2.transAxes)
f3_ax3.text(-0.1, 1, "(e)", fontsize=14, va="bottom", ha="right", transform=f3_ax3.transAxes)
f3_ax5.text2D(-0.1, 1, "(d)", fontsize=14, va="bottom", ha="right", transform=f3_ax5.transAxes)
f3_ax4.text(-0.1, 1, "(c)", fontsize=14, va="bottom", ha="right", transform=f3_ax4.transAxes)
f3_ax2.annotate('', xytext=(-2.5, 0.03), xy=(-1.5, 0.03),
            arrowprops=dict(color='b', arrowstyle="<-"))




for i, d in enumerate([828]): #199
    dat = load1dFig3(d)
    vg = dat[:,1]
    vt = dat[:,2]
    # vr = dat[:, 4]
    # curr = vt/(25813/3)
    # G = 3*vt/(vt+vr)
    # vs = (vt + vr)
    vsavg = 25813/3 * 5e-9
    G = 3*vt/vsavg
    Gmaxindex = np.argmax(G)
    f3_ax2.plot(1e3*(vg - vg[Gmaxindex]), G, 'b')
    

    dGfilter = savgol_filter(G, 30, 2, deriv=1, delta=np.diff(vg)[0])
    indexnoise = np.argmax(dGfilter)
    f3_ax2.plot(1e3*(vg[indexnoise]-vg[Gmaxindex]), G[indexnoise], 'o', color='purple', markersize=10)
    # ax2.plot(vg, np.gradient(curr)*1e12/np.diff(vg)[0], 'r')
    ax2.plot(1e3*(vg-vg[Gmaxindex]), dGfilter, 'r--')
    # print(vg[np.argmax(currfilter)])
f3_ax2.set_ylabel(r'$G$($e^2/h$)')
ax2.set_ylabel(r'$dG/dV_p$($e^2/h/V$)')
f3_ax2.set_xlabel(r'$\delta V_{pR}$ (mV)')
# ax.set_xlim(-0.5295, -0.5236)
# skunk.connect(f1_ax3, 'zoom')
# f3_ax3.set_axis_off()


firstfile = '231107_1000s_syncOFF_liaf88Hz_noiseonleftCBF_rplunger_fs300_tc3ms_FIRfilter_5nAlia860only_Vgm502.8mVR_iter0.txt.gz'
numfiles = 1
filelist = sortedfilelist(lia_dir)
startindex = filelist.index(firstfile)
fs = 305.18
alpha = 0.18/6
slope = 0.0017
time = np.arange(0, 1000, 1/fs)
chargenoise_psd = np.zeros((numfiles, int((1/1.00001638e-03)*fs/2) + 1))
counter = 0
chargenoisesum = np.zeros((int((1/1.00001638e-03)*fs/2) + 1,))
for peaknum in range(numfiles):


    if peaknum%1==0:
        dat = np.loadtxt('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum])
        sigma = np.std(dat)

        with gzip.open('../20231026_HMIA13_6 QD/data//lia/' + filelist[startindex+peaknum],'r') as f:
            firstline = f.readline()
            header = str(firstline).split(',')
            slope = float(header[3])
            print(slope)
        f.close()

        if 1e9*sigma < 300:
            flag=1
            counter+=1
        else:
            flag=0

        if peaknum == 0:
            cutoff = 20  # Desired cutoff frequency in Hz
            numtaps = 331  # Filter order + 1 (adjust as needed)
            # --- Design FIR filter ---
            # Normalized cutoff: cutoff / (Nyquist frequency)
            nyq = fs / 2
            fir_coeff = firwin(numtaps, cutoff / nyq)
            filterednoise = lfilter(fir_coeff, 1.0, 3*dat/(vsavg))
            f3_ax4.plot(time[:len(dat)], 3*dat/(vsavg), lw=1, label='raw', color=(0.9,0.8,0.9))
            f3_ax4.plot(time[165:len(dat)], filterednoise[165:], label='filtered', color=(0.5,0,0.5))

        
        print(1e9*sigma)
        print(flag)
        print('--')
        f, noise = noisepsd(dat, fs, rms=True)

        chargenoise_psd[peaknum, :] = noise*((25813/3)**2)*(alpha**2)/(slope**2)
        chargenoisesum = chargenoisesum + flag*chargenoise_psd[peaknum, :]
        # ax2.semilogy(f, chargenoise_psd[peaknum, :])


# avgchargenoise_left = chargenoisesum/counter
f3_ax4.set_xlabel('time (seconds)')
f3_ax4.set_ylabel('$G$ ($e^2/h$)')
f3_ax4.legend()
# f3_ax42 = f3_ax4.twiny()

# for i, d in enumerate([829]):
#     dat = load1dFig3(d)
#     time = dat[:,0] - dat[0,0]
#     vt = dat[:,1]
#     # vr = dat[:, 4]
#     curr = vt/(25813/3)
#     G = 3*vt/vsavg
#     # vs = (vt + vr)
#     # vsavg = 25813/3 * 5e-9
#     # Gmaxindex = np.argmax(G)
#     f3_ax42.plot(time, G, color='b')
    


f3_ax3.loglog(f[1:], 1*avgchargenoise_left[1:], 'b', label='CPF')
# f3_ax3.loglog(f, 1*avgchargenoise_right, 'r--')

# ax2.set_xscale('linear')
# # ax2.set_xscale('linear')
# ax2.set_xlim(5,10)

# harmonic = 2.3
# i=1
# for i in range(14):
#     ax2.axvline(i*harmonic, ls ='--', color='gray')

alpha = 0.18/6
chargenoise_psd = noiseCPT*((25813/3)**2)*(alpha**2)#/(slope**2)

# ax2.loglog(flow, chargenoise_psd,'.-')

chargenoise_psd2 = noiseCPT2*((25813/3)**2)*(alpha**2)#/(slope**2)

# ax2.loglog(flow2, chargenoise_psd2,'.-')

# ax2.loglog(flow2[1:], 1e-13/flow2[1:]**2, ls='--')



chargenoise_psd3 = noiseCPT3*((25813/3)**2)*(alpha**2)#/(slope**2)

chargenoise_psd4 = noiseCPT4*((25813/3)**2)*(alpha**2)#/(slope**2)

chargenoise_psd5 = noiseCPT5*((25813/3)**2)*(alpha**2)#/(slope**2)

# ax2.loglog(flow3, chargenoise_psd3,'k.-')
# # ax2.loglog(flow4, chargenoise_psd4,'g.-')
# f3_ax3.loglog(flow, chargenoise_psd,'k.-', label='CPT, one-way with step')


a, b = np.polyfit(np.log(f[1:1000]), np.log(avgchargenoise_left[1:1000]), 1)
print(a)
print(np.exp(b))
f3_ax3.loglog(f[1:], np.exp(b)/f[1:]**(-1*a), ls='--', color='gray')
f3_ax3.annotate(f"\u03b2 = {-1*a:.2f}", (0.1,1e-14), color='blue',fontsize=16)

f3_ax3.loglog(flow5, chargenoiseCPT,'r.-', label='CPT')
a, b = np.polyfit(np.log(flow5[1:]), np.log(chargenoiseCPT[1:]), 1)
f3_ax3.loglog(flow5[1:], np.exp(b)/flow5[1:]**(-1*a), ls='--', color='k')
print(a)
print(np.exp(b))
# f3_ax3.loglog(flow3[1:], np.exp(b)/flow3[1:]**(-1*a),'k--')

f3_ax3.set_xlim(2e-4,20)
f3_ax3.set_ylim(1e-15, 1e-6)
f3_ax3.set_ylabel('$S_{\mu} (eV^2/Hz)$')
f3_ax3.set_xlabel('f (Hz)')
f3_ax3.grid(ls='--', lw=0.4)
f3_ax3.legend()

f3_ax3.annotate(f"\u03b2 = {-1*a:.2f}", (0.005,1e-8), color='red',fontsize=16)



b = np.linspace(0,1,1001)
# ax2 = ax.twinx()
vgpeak = np.zeros((500,))
peak_temps = np.zeros((500,))
peak_temps_err = np.zeros((500,))
time = np.flip(np.linspace(41, 41*500, 500)/3600)
vtfilter = np.zeros((61,500))
vg = np.zeros((61,500))
maxidx = np.zeros((500,))
alpha = 0.18/6 ## lever arm
kT = 4.05e-6
k = 8.6e-5
for i,d in enumerate(np.arange(3504, 2504,-2)):
    dat  = load1dFig3(d)
    vt = dat[:,2]
    vg[:,i] = dat[:,1]


    # ax.plot(vg, vt)
    vtfilter[:, i] = savgol_filter(vt, 20, 2, deriv=0, delta=np.diff(vg[:,i])[0])
    maxidx[i] = np.argmax(vtfilter[:, i])
    f3_ax5.plot(vg[int(maxidx[i]),i], time[i],0, 'o', markersize=1, color='white', zorder=9 )

    if i%200==0:
        # f3_ax5.plot(time[i], vg[:,i], 1e6*vtfilter[:,i], lw=0.5, color='k', zorder=10)
        for j in range(0, 61, 1):
            # f3_ax5.scatter(vg[j,i], time[i], 1e6*vtfilter[j,i], marker='o', s=10, color=plt.cm.cividis((vtfilter[j,i] - vtfilter.min()) / (vtfilter.max() - vtfilter.min())), zorder=10)
    # f3_ax5.plot(vg[int(maxidx[i]),i], time[i], 0.0, color='r', marker='x', zorder=3)

            params['center'].set(value = vg[int(maxidx[i]), i], min = vg[0, i], max = vg[-1, i])
            params['Vt0'].set(value = vt[int(maxidx[i])], min = 0, max = 10*vt[int(maxidx[i])])
            params['x'].set(value=alpha/kT, min = 0, max = 10*alpha/kT)

            result = cbmodel.fit(vt, params, Vg = vg[:,i])
            peak_temps[i] = alpha/(k*result.params['x'].value)
            peak_temps_err[i] = peak_temps[i]*result.params['x'].stderr/result.params['x'].value
            vgpeak[i] = result.params['center'].value

            f3_ax5.plot(vg[:,i], time[i], 3*cbsinh(vg[:,i], vgpeak[i], vt[int(maxidx[i])], result.params['x'])/(0.1*vsavg), color='gray', ls='--', zorder=10)
            f3_ax5.plot(vg[j,i], time[i], 3*vtfilter[j,i]/(0.1*vsavg), marker='o', color=plt.cm.plasma((vtfilter[j,i] - vtfilter.min()) / (vtfilter.max() - vtfilter.min())), zorder=10)

# tmesh, vpmesh = np.meshgrid(time, vg)
# for i in range(vtfilter.shape[1]):
z_proj = np.full_like(vg,0)
f3_ax5.plot_surface(vg, np.full_like(vg, time), z_proj, facecolors=plt.cm.plasma((vtfilter - vtfilter.min()) / (vtfilter.max() - vtfilter.min())),
                rstride=1, cstride=1, shade=False, zorder=1, alpha=0.8)

f3_ax5.view_init(elev=30, azim=315, roll=0)
# for i in range(500):
#     f3_ax5.plot(vg[int(maxidx[i]),i], time[i], 0.2, color='r', marker='x', zorder=10)
# f3_ax5.plot(vg[0, :], time[:], 0., color='r', lw=2, zorder=10)

# f3_ax5.pcolormesh(np.full_like(vg, time), vg, vtfilter, cmap='inferno', vmin=0, vmax=1e-7)
# f3_ax5.pcolormesh(tmesh, vpmesh, vtfilter, cmap='inferno', zorder=0)
# f3_ax4.plot(time[:i]/3600, vgpeak[:i])
f3_ax5.set_ylabel('time (hours)')
# f3_ax4.set_xlabel('time (hours)')

f3_ax5.set_zlim(0,0.25)

# f3_ax4.set_ylabel('$V_{peak}$ (V)')
f3_ax5.set_xlabel('$V_{pR}$ (V)')
f3_ax5.set_zlabel('$G$ ($e^2/h$)')

# ax2.errorbar(time[:i], 1e3*peak_temps[:i], yerr = 1e3*peak_temps_err[:i], fmt = 'r.')

    # time[i] = dat[-1,0]
    # timdur[i] = time[i] - dat[0,0]
    # ax.plot(vg, vt)



skunk.connect(f3_ax1, 'sk2')
# f3_ax4.grid(ls='--', lw=0.4)

#svg = skunk.pltsvg(fig=fig2)
svg = skunk.insert(
    {  
        'sk2': 'ChargeNoiseFig4apaper.svg'
            
    })

# # fig2.set_constrained_layout(False)
# # fig3.tight_layout()

# # fig1.canvas.draw()
# # # we want the legend included in the bbox_inches='tight' calcs.
# # # cbar1.set_in_layout(True)
# # # cbar2.set_in_layout(True)
# # # we don't want the layout to change at this point.
# #fig1.tight_layout()

skunk.display(svg)
